# SENSERO – Stage 1: Create Dataset GeoPackage

Traverse the SENSERO dataset tree under `GeoTiff/Patch_336/.../Multispectral/`
and build a single **GeoPackage** containing the valid-data contour of every patch.

Each feature carries:

| Attribute | Description |
|-----------|-------------|
| `filename` | Basename of the `.tif` |
| `rel_path` | Path relative to the dataset root |
| `subfolder` | Parent folder name (tile / location id) |
| `width_px` | Raster width in pixels |
| `height_px` | Raster height in pixels |
| `bands` | Number of spectral bands |
| `crs_src` | Original CRS of the file |
| `area_m2` | Approximate footprint area in m² |

Output CRS is **EPSG:4326** (WGS-84).  
The attribute table will be extended in subsequent stages.

**Requirements:** `pip install rasterio shapely fiona`

## Configuration

In [4]:
# ── CONFIG ─────────────────────────────────────────────────────────────
ROOT_DIR       = "/home/ubuntu/SENSERO/GeoTiff/Patch_336/"
OUTPUT_GPKG    = "/home/ubuntu/SENSERO/sensero_patches.gpkg"
TARGET_CRS     = "EPSG:4326"

# Set True to keep only 3360 × 3360 m patches; False to keep everything
FILTER_SIZE    = True
EXPECTED_SIZE_M = 3360
SIZE_TOLERANCE  = 5   # metres

## Imports

In [5]:
import math
import os
import sys

import fiona
import rasterio
from rasterio.features import shapes as rio_shapes
from rasterio.warp import transform_geom
from shapely.geometry import Polygon, MultiPolygon, mapping, shape
from shapely.ops import unary_union
from shapely.validation import make_valid

## Helper functions

In [6]:
# Approximate degree-to-metre factors at ~45°N (Romania)
_DEG2M_LON = math.cos(math.radians(45)) * 111_320
_DEG2M_LAT = 111_320


def approx_area_m2(geom_wgs84) -> float:
    """Rough area in m² for a WGS-84 geometry at Romanian latitudes."""
    return geom_wgs84.area * _DEG2M_LON * _DEG2M_LAT


def patch_is_expected_size(
    src: rasterio.DatasetReader,
    expected: float = EXPECTED_SIZE_M,
    tolerance: float = SIZE_TOLERANCE,
) -> bool:
    """Return True if the raster footprint is ~expected × expected metres."""
    if not src.crs or src.crs.is_geographic:
        return True  # can't verify — keep it
    rx, ry = src.res
    w = abs(rx) * src.width
    h = abs(ry) * src.height
    return abs(w - expected) <= tolerance and abs(h - expected) <= tolerance


def valid_polygon(src: rasterio.DatasetReader, target_crs: str = TARGET_CRS):
    """
    Return a (Multi)Polygon of valid (non-nodata) pixels reprojected to
    *target_crs*, or None on failure.
    """
    mask = src.read_masks(1)  # 255 = valid, 0 = nodata
    parts: list = []
    for geom, val in rio_shapes(mask, mask=mask, transform=src.transform):
        if val == 0:
            continue
        if src.crs is not None and src.crs.to_string() != target_crs:
            geom = transform_geom(src.crs, target_crs, geom, precision=8)
        try:
            s = make_valid(shape(geom).buffer(0))
            if not s.is_empty:
                parts.append(s)
        except Exception:
            pass

    if not parts:
        return None

    union = make_valid(unary_union(parts).buffer(0))

    # Retain only polygonal components
    if union.geom_type == "GeometryCollection":
        polys = [g for g in union.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if not polys:
            return None
        union = unary_union(polys)

    return union

## GeoPackage schema

In [7]:
SCHEMA = {
    "geometry": "Polygon",
    "properties": {
        "filename":  "str",
        "rel_path":  "str",
        "subfolder": "str",
        "width_px":  "int",
        "height_px": "int",
        "bands":     "int",
        "crs_src":   "str",
        "area_m2":   "float",
    },
}

## Collect patch footprints

In [8]:
abs_root = os.path.abspath(ROOT_DIR)
features: list[dict] = []
skipped = 0
errors  = 0

for dirpath, dirnames, filenames in os.walk(abs_root):
    dirnames[:] = [d for d in dirnames if d != ".ipynb_checkpoints"]

    # Only process inside Multispectral folders
    if "multispectral" not in [p.lower() for p in dirpath.split(os.sep)]:
        continue

    for fname in sorted(filenames):
        if not fname.lower().endswith((".tif", ".tiff")):
            continue

        tif_path  = os.path.join(dirpath, fname)
        rel_path  = os.path.relpath(tif_path, abs_root)
        # One level above Multispectral → tile / location id
        subfolder = os.path.basename(os.path.dirname(dirpath))

        try:
            with rasterio.open(tif_path) as src:
                if FILTER_SIZE and not patch_is_expected_size(src):
                    skipped += 1
                    continue

                crs_src = src.crs.to_string() if src.crs else "unknown"
                w_px    = src.width
                h_px    = src.height
                n_bands = src.count
                poly    = valid_polygon(src)

            if poly is None:
                print(f"[WARN] no valid polygon: {rel_path}")
                skipped += 1
                continue

            features.append({
                "geometry": mapping(poly),
                "properties": {
                    "filename":  fname,
                    "rel_path":  rel_path,
                    "subfolder": subfolder,
                    "width_px":  w_px,
                    "height_px": h_px,
                    "bands":     n_bands,
                    "crs_src":   crs_src,
                    "area_m2":   round(approx_area_m2(poly), 1),
                },
            })

        except Exception as exc:
            print(f"[ERROR] {rel_path}: {exc}")
            errors += 1

print(f"Collected {len(features)} patch contours "
      f"(skipped {skipped}, errors {errors})")

Collected 10000 patch contours (skipped 0, errors 0)


## Write GeoPackage

In [9]:
if not features:
    print("Nothing to write.")
else:
    os.makedirs(os.path.dirname(OUTPUT_GPKG) or ".", exist_ok=True)

    with fiona.open(
        OUTPUT_GPKG,
        "w",
        driver="GPKG",
        crs=TARGET_CRS,
        schema=SCHEMA,
    ) as dst:
        for feat in features:
            dst.write(feat)

    print(f"GeoPackage saved → {OUTPUT_GPKG}")
    print(f"  ({len(features)} features, CRS={TARGET_CRS})")
    print("✓ Done.")

GeoPackage saved → /home/ubuntu/SENSERO/sensero_patches.gpkg
  (10000 features, CRS=EPSG:4326)
✓ Done.
